# 04. Feature Engineering & Preprocessing Pipeline

## Executive Summary
Machine learning algorithms require numerical representations. This notebook constructs a scikit-learn `ColumnTransformer` preprocessing pipeline ensuring zero data leakage between train and test splits.

### Principles Followed:
1. **Stratified Splitting:** 80/20 train/test split stratified on `Churn` to preserve the 26.54% positive class ratio.
2. **Leakage Prevention:** Preprocessor fits strictly on `X_train` and transforms `X_test`.
3. **Pipeline Structure:**
   - Numerical (`SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges`): `StandardScaler()`
   - Categorical (15 features): `OneHotEncoder(handle_unknown='ignore')`
4. **Serialization:** Save `preprocessor.pkl` and split CSVs to disk.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

pd.set_option("display.max_columns", None)

data_path = Path("../data/processed/churn_cleaned.csv")
df = pd.read_csv(data_path)
print(f"Loaded cleaned data: {df.shape}")

Loaded cleaned data: (7043, 20)


## 1. Feature and Target Segregation

In [2]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape  :", y.shape)

Features shape: (7043, 19)
Target shape  : (7043,)


## 2. Feature Schema Definition

In [3]:
numerical_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = X.select_dtypes(include="object").columns.tolist()

print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

Numerical features (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 3. Stratified Train-Test Split (80 / 20)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape} | y_test shape: {y_test.shape}")
print("\nTrain target distribution:\n", y_train.value_counts(normalize=True).round(4))
print("\nTest target distribution:\n", y_test.value_counts(normalize=True).round(4))

X_train shape: (5634, 19) | X_test shape: (1409, 19)
y_train shape: (5634,) | y_test shape: (1409,)

Train target distribution:
 Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64

Test target distribution:
 Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64


## 4. Constructing Scikit-Learn Preprocessing Pipeline

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ],
    remainder="drop"
)

# Fit strictly on X_train to prevent data leakage
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Transformed features count: {len(feature_names)}")
print("First 10 transformed feature names:")
print(feature_names[:10])

Transformed features count: 45
First 10 transformed feature names:
['num__SeniorCitizen' 'num__tenure' 'num__MonthlyCharges'
 'num__TotalCharges' 'cat__gender_Female' 'cat__gender_Male'
 'cat__Partner_No' 'cat__Partner_Yes' 'cat__Dependents_No'
 'cat__Dependents_Yes']


## 5. Reconstructing Formatted DataFrames

In [6]:
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

print("X_train_df dimensions:", X_train_df.shape)
print("Missing values in X_train_df:", X_train_df.isnull().sum().sum())
X_train_df.head(3)

X_train_df dimensions: (5634, 45)
Missing values in X_train_df: 0


,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,cat__gender_Female,cat__gender_Male,cat__Partner_No,cat__Partner_Yes,cat__Dependents_No,cat__Dependents_Yes,cat__PhoneService_No,cat__PhoneService_Yes,cat__MultipleLines_No,cat__MultipleLines_No phone service,cat__MultipleLines_Yes,cat__InternetService_DSL,cat__InternetService_Fiber optic,cat__InternetService_No,cat__OnlineSecurity_No,cat__OnlineSecurity_No internet service,cat__OnlineSecurity_Yes,cat__OnlineBackup_No,cat__OnlineBackup_No internet service,cat__OnlineBackup_Yes,cat__DeviceProtection_No,cat__DeviceProtection_No internet service,cat__DeviceProtection_Yes,cat__TechSupport_No,cat__TechSupport_No internet service,cat__TechSupport_Yes,cat__StreamingTV_No,cat__StreamingTV_No internet service,cat__StreamingTV_Yes,cat__StreamingMovies_No,cat__StreamingMovies_No internet service,cat__StreamingMovies_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_No,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check
0,-0.441773,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,-0.441773,-0.711743,0.337478,-0.503635,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,-0.441773,-0.793155,-0.809013,-0.749883,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0


## 6. Serializing Pipeline Artifacts

In [7]:
models_dir = Path("../models")
data_dir = Path("../data/processed")
models_dir.mkdir(parents=True, exist_ok=True)

# Save preprocessor artifact
joblib.dump(preprocessor, models_dir / "preprocessor.pkl")
print(f"Preprocessor saved to {models_dir / 'preprocessor.pkl'}")

# Save CSV datasets
X_train_df.to_csv(data_dir / "X_train.csv", index=False)
X_test_df.to_csv(data_dir / "X_test.csv", index=False)
pd.DataFrame({"Churn": y_train}).to_csv(data_dir / "y_train.csv", index=False)
pd.DataFrame({"Churn": y_test}).to_csv(data_dir / "y_test.csv", index=False)

print("Transformed training and test CSVs saved successfully.")

Preprocessor saved to ..\models\preprocessor.pkl


Transformed training and test CSVs saved successfully.

## Summary Checklist
- [x] Performed 80/20 stratified split (5,634 train, 1,409 test records).
- [x] Fitted ColumnTransformer strictly on `X_train` with `StandardScaler` and `OneHotEncoder`.
- [x] Generated 45 standardized features with clean names.
- [x] Serialized `preprocessor.pkl` and dataset CSVs for model training.